In [142]:
import pandas as pd
import numpy as np
import re
import string
from pathlib import Path
from config.feature_map import get_dataset
from hazm import word_tokenize

In [143]:
class TextPreprocessing:
    def __init__(self):
        pass

    # @staticmethod
    def Overview(self, df):
        print("Overview : \n" , df.head())
        print("\nIs Null : \n" , df.isna().sum())

    # @staticmethod
    def create_raw_text(self, df, columns):
        raw_text = (
            df[columns]
            .fillna("")
            .astype(str)
            .agg(" ".join, axis=1)
            .str.strip()
        )
        raw_text = raw_text.map(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
        print("\nRaw Text Samples : \n", raw_text.head())
        return raw_text
    
    # @staticmethod
    def Normalize_text(self, input_text):

        if isinstance(input_text, pd.Series):
            normalized = input_text.astype(str).copy()
        else:
            normalized = pd.Series([str(input_text)])

        replacements = {
            "ۀ": "ه",
            "ك": "ک",
            "ي": "ی",
            "ئ": "ی",
            "أ": "ا",
            "إ": "ا",
            "آ": "ا",
            "ؤ": "و",
            "\u200c": " ",
            "\u00ad": " "
        }

        for old, new in replacements.items():
            normalized = normalized.str.replace(old, new, regex=False)
        normalized = normalized.str.replace(r"<[^>]+>", " ", regex=True) #HTML
        normalized = normalized.str.replace(r"http\S+|www\S+", " ", regex=True) #URL
        normalized = normalized.str.replace(r"\S+@\S+", " ", regex=True) #Email
        punctuation = "،؛؟«»" + string.punctuation
        normalized = normalized.str.replace("[" + re.escape(punctuation) + "]", " ", regex=True)
        normalized = normalized.apply(lambda x: re.sub(r'(.)\1{2,}', r'\1', x) 
                                      if isinstance(x, str) else x) #Repeated Characters
        normalized = normalized.str.replace(r"\s+", " ", regex=True) #Spaces
        normalized = normalized.str.strip()

        print("\nNormalized Samples:\n")
        for i, text in enumerate(normalized.head(3), start=1):
            print(f"{i}. {text[:100]}")

        return normalized
    
    # @staticmethod
    def Tokenize(self, input_text):
        if isinstance(input_text, pd.Series):
            return input_text.apply(word_tokenize)
        return word_tokenize(str(input_text))
    
    # @staticmethod
    # def Export(self, df, output_path):
    #     output_path = Path(output_path)
    #     output_path.parent.mkdir(
    #         parents=True,
    #         exist_ok=True
    #     )
    #     if output_path.suffix == ".csv":
    #         df.to_csv(
    #             output_path,
    #             index=False,
    #             encoding="utf-8-sig"
    #         )
    #     elif output_path.suffix in [".xlsx", ".xls"]:
    #         df.to_excel(
    #             output_path,
    #             index=False
    #         )
    #     print(f"\nDataset Saved : {output_path}")

In [144]:
TextPre = TextPreprocessing()

DATA_DIR = Path("Dataset")
files = list(DATA_DIR.glob("*"))

for file_path in files:
    dataset_name = file_path.stem
    config = get_dataset(dataset_name)
    if not config:
        print(f"{dataset_name} not found in feature_map")
        continue

    if file_path.suffix == ".csv":
        df = pd.read_csv(file_path)
    elif file_path.suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)
    else:
        continue

    TextPre.Overview(df)
    df["raw_text"] = TextPre.create_raw_text(df, config["preprocess_columns"])

    df["raw_text_normalized"] = TextPre.Normalize_text(df["raw_text"])

    df["tokens"] = TextPre.Tokenize(df["raw_text_normalized"])
    
    # output_dir = Path("Dataset/Clean")
    # output_file = output_dir / file_path.name
    # TextPre.Export(df, output_file)

comment not found in feature_map
Overview : 
    product_id                                      product_title title_en  \
0        3692                        ماوس بی‌سیم لاجیتک مدل M325       IT   
1       90213  شارژر همراه شیاومی مدل NDY-02-AN با ظرفیت 1000...       AC   
2       59473              یدک پولیشر میکروفایبر مهسان مدل 20119       HW   
3      120499   گوشی موبایل هوآوی آنر مدل 5X KIW-L21 دو سیم‌کارت       MO   
4       67200  شارژر همراه شیائومی مدل Mi ظرفیت 5000 میلی آمپ...       AC   

   user_id  likes  dislikes verification_status        recommend  \
0   989472      0         0            verified               \N   
1  3862150      4         1            verified      recommended   
2   626843      1         0            verified  not_recommended   
3   786887      6        11            verified          no_idea   
4   854531     19         4            verified          no_idea   

                                 title  \
0                                  NaN  

KeyboardInterrupt: 